# CUAD transformer pipeline

This notebook is the single end-to-end entry point for preparing CUAD chunks, training the DistilRoBERTa classifier, and evaluating it. Run it from the repository root or from `notebooks/`. Training is intended for Google Colab with a GPU.

**Text and label order:** reshape/clean the annotation data; chunk the original contract text; map CUAD answer spans onto those original offsets; add the light cleaner's `text_light` column; train and evaluate. We do not strip stop words or normalize contract text before chunking, because span offsets refer to the original text.

The annotation artifact retains all 41 CUAD categories. The model has **34 output labels**: the categories with positive training labels, including the two rare categories the team chose to retain. Seven categories below the 10% cutoff have no training rows and are not model outputs. The official 408/102 contract split is preserved.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in candidates if (path / "data/cuad/train_separate_questions.json").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Device:", "CUDA" if torch.cuda.is_available() else "MPS" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "CPU")


## 1. Load and prepare CUAD annotations

This reshapes the question/answer JSON and applies the agreed category cutoff to training records. It does not alter the original contract text. Test annotations retain all 41 categories so source labels remain auditable.


In [ ]:
with (DATA_DIR / "cuad/train_separate_questions.json").open() as f:
    train_data = json.load(f)
with (DATA_DIR / "cuad/test.json").open() as f:
    test_data = json.load(f)

print("Train contracts:", len(train_data["data"]))
print("Test contracts:", len(test_data["data"]))


In [ ]:
from src.ingest.preprocessing import prepare_contract_categories

categories, category_summary = prepare_contract_categories(
    train_data,
    test_data,
    cutoff=0.10,
)
print("Category preparation:", category_summary)


## 2. Chunk the original contract text

The chunker uses DistilRoBERTa's tokenizer to make 510-token text windows with 128-token overlap. It returns text plus original character offsets. We do not clean contract text before this step because the CUAD answer offsets refer to the original text.


In [ ]:
from src.ingest.chunker import get_tokenizer, MAX_TOKENS, OVERLAP
from src.ingest.preprocessing import build_chunks

chunks = build_chunks(train_data, test_data, tokenizer=get_tokenizer())
print("Chunk settings:", MAX_TOKENS, "tokens /", OVERLAP, "overlap")
print("Chunks:", len(chunks), chunks.groupby("split").size().to_dict())


## 3. Map annotated spans to chunk labels

Labels are assigned by overlap with CUAD's original answer spans. This creates the 41 nullable source-label columns; categories removed from train remain missing there. The model output list is derived from positive labels in the train split and should contain 34 categories.


In [ ]:
from src.ingest.label_chunks import label_chunks
from src.models.transformer import select_train_positive_categories

label_result = label_chunks(chunks, categories, rule="any_overlap")
chunk_labels = label_result.chunk_labels
model_categories = select_train_positive_categories(chunk_labels)
print("Source label columns:", len([c for c in chunk_labels.columns if c not in {"split", "contract_id", "chunk_id", "chunk_index", "start_char", "end_char", "n_tokens"}]))
print("Model output categories:", len(model_categories))
assert len(model_categories) == 34


## 4. Clean model text after span alignment

This is where the light transformer cleaner is used. It adds `text_light` by blanking known extraction artifacts while preserving character length. The original `text`, chunk offsets, and span labels remain available. The model reads `text_light`.


In [ ]:
from src.clean.light import add_light_cleaned_text

chunks = add_light_cleaned_text(chunks, text_column="text", output_column="text_light")
assert chunks["text_light"].str.len().equals(chunks["text"].str.len())
print("Chunk text columns:", [column for column in ("text", "text_light") if column in chunks.columns])


## 5. Save reproducible preprocessing artifacts

`chunks.parquet` keeps both original `text` and cleaned `text_light`. `chunk_labels.parquet` keeps all 41 nullable source categories; the model selects the 34 train-supported categories from that label matrix. Span audit files make label placement inspectable.


In [ ]:
categories.to_json(
    PROCESSED_DIR / "contract_categories.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
chunks.to_parquet(PROCESSED_DIR / "chunks.parquet", index=False)
chunk_labels.to_parquet(PROCESSED_DIR / "chunk_labels.parquet", index=False)
label_result.span_overlaps.to_parquet(PROCESSED_DIR / "span_overlaps.parquet", index=False)
label_result.span_audit.to_parquet(PROCESSED_DIR / "span_audit.parquet", index=False)
label_result.pair_status.to_parquet(PROCESSED_DIR / "pair_status.parquet", index=False)
print("Saved chunks, labels, and span audits under", PROCESSED_DIR)


## 6. Train and evaluate

The trainer uses the official train contracts and evaluates once on the official test contracts. It reports contract-level per-category precision, recall, F1, and positive support, plus macro-F1. The configured 0.5 threshold is fixed for this run; do not tune it on test results. Imbalance handling is configurable in `configs/transformer.yaml` and remains a team decision.


In [ ]:
from src.models.trainer import train_and_evaluate

run = train_and_evaluate(PROJECT_ROOT / "configs/transformer.yaml")
print("Training history:")
display(run["history"])
print("Per-category official test results:")
display(run["per_category_metrics"].sort_values("f1", ascending=False))
print("Summary:", run["summary"])
print("Saved run to:", run["output_dir"])


## Artifacts and interpretation

The model bundle records the backbone, tokenizer, classifier head, resolved 34-category list, thresholds, training settings, and test metrics. Chunk probabilities are max-pooled by contract/category before scoring so overlap does not count a clause twice. Macro-F1 weights each modeled category equally. Chunk-level predictions are saved separately for error analysis.
